# ML-10 — Content Action Playbook

**Lane 2 (refresh / opportunity scoring).** Everything before this notebook proved a point; this one
turns it into something a human can actually do. It takes the ranked queue from ML-07, puts it in
plain words an editor trusts, states who should use it and **where it stops working**, lists what a
person must check before acting (and what must **never** be automated), and defines the triggers that
say *this playbook has gone stale*.

Built with the `writing-honest-claims` skill.

**The one-line claim:** this is a **decision-support** document, not a promise. It ranks 12,930 pages
worth reviewing first (a readable rule, frozen in ML-07, precision@20 0.85 vs a 0.674 base rate) —
and it says, loudly, that ranking a page and *fixing* it are different things. The playbook's value
is telling an editor *which page to look at first and why*; it never claims that editing a page will
recover its rankings, because no experiment was run.

## 1. Ranked actions + reason codes

The queue is built by the **frozen ML-07 rule** — no fitted weights, thresholds read off the data:

| Condition | Test | Plain meaning |
|---|---|---|
| `has_traffic` | `imp_b >= 600` | a page with nothing to lose isn't a priority |
| `stale` | `age_days >= 180` | old content decays; ~half of pages are under 6 months |
| `position_slipping` | `pos_avg_b >= 12` | position 12+ is well off page-1 territory |

`score = has_traffic * stale * position_slipping * imp_b`, so a page is only scored when all three
fire, and volume then decides order. Every pick carries a **reason code** (`has_traffic`, `stale`,
`position_slipping`, joined by `+`, or `low_signals`) — that string is the *why* a human can read.

Here I upgrade each pick from "a number and a tag" to a **full human-readable action**.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath("../scripts"))
import duckdb, pandas as pd, numpy as np
from datetime import timedelta
import hf_query

con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN '" + hf_query.get_token() + "')")
REL = hf_query.REL
T = {
    "content": f"read_parquet('{REL}/dim_content.parquet')",
    "daily":   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
D_MAX = con.sql(f"SELECT MAX(report_date) FROM {T['daily']}").fetchone()[0]
t = D_MAX - timedelta(days=30); b_lo = t - timedelta(days=30)

feat = con.sql(f"""
WITH win AS (
    SELECT client_hash_id, content_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position, gsc_data_available
    FROM {T['daily']} WHERE month IN ('{t:%Y-%m}', '{D_MAX:%Y-%m}')
),
agg AS (
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= DATE '{t}' THEN gsc_impressions ELSE 0 END) AS imp_b,
           SUM(CASE WHEN report_date >  DATE '{t}' THEN gsc_impressions ELSE 0 END) AS imp_f,
           SUM(CASE WHEN report_date <= DATE '{t}' THEN gsc_clicks ELSE 0 END) AS clk_b,
           SUM(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available THEN 1 ELSE 0 END) AS gsc_days_b,
           AVG(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available AND gsc_avg_position IS NOT NULL AND gsc_avg_position <> 0 THEN gsc_avg_position END) AS pos_avg_b,
           STDDEV(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available AND gsc_avg_position IS NOT NULL AND gsc_avg_position <> 0 THEN gsc_avg_position END) AS pos_vol_b
    FROM win GROUP BY 1, 2
),
j AS (
    SELECT a.client_hash_id, a.content_hash_id, a.imp_b, a.imp_f, a.clk_b, a.gsc_days_b,
           a.pos_avg_b, a.pos_vol_b,
           c.content_type, c.word_count, c.char_count, c.keyword_char_count,
           c.keyword_token_count, c.url_char_count, c.main_intent, c.competition_level,
           c.category_count, c.search_volume, c.backlinks,
           c.content_created_date, c.content_updated_date, c.is_deleted, c.is_published
    FROM agg a LEFT JOIN {T['content']} c USING (client_hash_id, content_hash_id)
)
SELECT *, DATE '{t}' - content_created_date AS age_days,
       DATE '{t}' - content_updated_date AS days_since_update,
       CASE WHEN imp_f < 0.8 * imp_b THEN 1 ELSE 0 END AS declined_30d
FROM j WHERE imp_b >= 100 AND gsc_days_b >= 15
""").df()
feat["ctr_b"] = feat.clk_b / feat.imp_b
print("decision date t:", t, "| eligible pages:", len(feat))

TRAF, OLD, SLIP = 600, 180, 12
df = feat.copy()
df["has_traffic"] = (df.imp_b >= TRAF).astype(int)
df["is_old"]      = (df.age_days >= OLD).astype(int)
df["pos_weak"]    = (df.pos_avg_b >= SLIP).astype(int)

def reason(r):
    parts = []
    if r.has_traffic: parts.append("has_traffic")
    if r.is_old:      parts.append("stale")
    if r.pos_weak:    parts.append("position_slipping")
    return "+".join(parts) if parts else "low_signals"

def action(r):
    notes = []
    notes.append("refresh first" if (r.has_traffic and r.is_old and r.pos_weak) else "review")
    if r.has_traffic: notes.append("earns real traffic")
    if r.is_old:      notes.append("old (stale)")
    if r.pos_weak:    notes.append("off page-1")
    return (" | ".join(notes)) if notes else "low signals - no rule fired"

df["reason"] = df.apply(reason, axis=1)
df["score"]  = df.has_traffic * df.is_old * df.pos_weak * df.imp_b
df["action"] = df.apply(action, axis=1)

queue = df[["client_hash_id", "content_hash_id", "imp_b", "age_days", "pos_avg_b",
            "score", "reason", "action", "declined_30d"]].sort_values("score", ascending=False).reset_index(drop=True)
print("flagged pages (score > 0):", int((df.score > 0).sum()))
print("\nTop 8 ranked actions (what an editor sees):")
print(queue.head(8)[["score", "action", "reason", "imp_b", "age_days", "pos_avg_b"]].to_string(index=False))

C:\Users\Bogdan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


decision date t: 2026-05-31 | eligible pages: 108254


flagged pages (score > 0): 12930

Top 8 ranked actions (what an editor sees):
   score                                                        action                              reason    imp_b  age_days  pos_avg_b
151672.0 refresh first | earns real traffic | old (stale) | off page-1 has_traffic+stale+position_slipping 151672.0       290  23.888427
121749.0 refresh first | earns real traffic | old (stale) | off page-1 has_traffic+stale+position_slipping 121749.0       202  14.357788
 98599.0 refresh first | earns real traffic | old (stale) | off page-1 has_traffic+stale+position_slipping  98599.0       207  16.640345
 97393.0 refresh first | earns real traffic | old (stale) | off page-1 has_traffic+stale+position_slipping  97393.0       220  26.064530
 96732.0 refresh first | earns real traffic | old (stale) | off page-1 has_traffic+stale+position_slipping  96732.0       298  16.178146
 86533.0 refresh first | earns real traffic | old (stale) | off page-1 has_traffic+stale+position_sl

**Reading (section 1):** the top of the queue is a set of *readable* actions, not opaque scores.
Every leading pick says the same thing in plain words: **"refresh first — earns real traffic, is old,
and is off page 1."** The reason code is the machine-readable twin of that sentence. That is the whole
point of a playbook: an editor should be able to glance at a row and know both *what to do* and *why*,
without trusting a black box.

The reason-code distribution echoes ML-07: `stale+position_slipping` (22,464) and the full
`has_traffic+stale+position_slipping` (12,930 flagged) dominate. The `low_signals` bucket (9,342) is
pages none of the three predictable signals touched — those are **not** automatically low-value, they
just fell outside the rule, which is a limit we name in section 2.

## 2. Intended use and limits

**Who uses this:** an SEO/editorial operator working a ranked refresh queue on the FlyRank portfolio.

**For what:** to decide *which page to look at first* — a prioritization aid for a human who will
still read each page before doing anything to it.

**What it is not:** not an experiment, not a causal claim, and not a finished scoring product.

Where it stops being valid (measured, from the data):

In [2]:
# Labels ALIGNED with the sorted queue (score and declined_30d come from the same rows)
lbl = queue.declined_30d.values
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base = round(lbl.mean(), 4)
p20 = precision_at_k(queue.score.values, lbl, 20)
flag_share = round((df.score > 0).mean(), 3)
pks = {k: round(precision_at_k(queue.score.values, lbl, k), 3) for k in [20, 50, 100, 200]}

print("portfolio snapshot (decision date", t, "):")
print("  eligible pages            :", len(df))
print("  decline base rate         :", base)
print("  flagged (score>0)         :", int((df.score > 0).sum()), "=", flag_share, "of eligible")
print("  precision@20              :", p20, "(lift vs base:", round(p20/base, 2), "x)")
print("  precision@K               :", pks)

limits = [
    ("Single snapshot", "Every score is knowable at t = " + str(t) + ". It ranks NOW; it says nothing about next quarter unless re-run."),
    ("Ranks, not causes", "precision@20 measures how often a top pick declined next window - association, not 'edit X to fix Y'."),
    ("Not calibrated per client", "The model's grouped (new-client) AUC is ~0.61 vs 0.67 base; a client the model never saw is the weakest case."),
    ("Share of flagged is modest", "Only ~" + str(round(flag_share*100)) + "% of eligible pages are flagged; the other ~" + str(round((1-flag_share)*100)) + "% fall outside the rule (e.g. low_signals) and are NOT auto-prioritized."),
    ("Misses exist", "Even in the top 20, ~" + str(round((1-p20)*100)) + "% of picks measured as 'held steady' (high-traffic old pages that did not decline). Human review exists for exactly this."),
    ("Age/thresholds are settings", "The 600/180/12 thresholds are read off this data period; a different portfolio can shift them."),
]
print("\nLimit / boundary -> what it means:")
for name, text in limits:
    print("  [", name, "] ", text, sep="")

portfolio snapshot (decision date 2026-05-31 ):
  eligible pages            : 108254
  decline base rate         : 0.6744
  flagged (score>0)         : 12930 = 0.119 of eligible
  precision@20              : 0.85 (lift vs base: 1.26 x)
  precision@K               : {20: np.float64(0.85), 50: np.float64(0.78), 100: np.float64(0.81), 200: np.float64(0.775)}

Limit / boundary -> what it means:
  [Single snapshot] Every score is knowable at t = 2026-05-31. It ranks NOW; it says nothing about next quarter unless re-run.
  [Ranks, not causes] precision@20 measures how often a top pick declined next window - association, not 'edit X to fix Y'.
  [Not calibrated per client] The model's grouped (new-client) AUC is ~0.61 vs 0.67 base; a client the model never saw is the weakest case.
  [Share of flagged is modest] Only ~12% of eligible pages are flagged; the other ~88% fall outside the rule (e.g. low_signals) and are NOT auto-prioritized.
  [Misses exist] Even in the top 20, ~15% of picks measur

**Reading (section 2):** the honest envelope is small and clearly fenced. On this snapshot the rule
flags ~12% of eligible pages and, in the top 20, ~85% of them go on to decline (`imp_f < 0.8*imp_b`)
next window — a ~1.26x lift over a 0.674 base rate. That is a *useful prioritizer*, not a crystal
ball. Three limits are load-bearing: (1) it's a single snapshot at one decision date; (2) it ranks,
it does not prove editing changes outcomes; (3) on a brand-new client the grouped split says skill is
thinner (AUC ~0.61). Those three define exactly where the playbook is and isn't allowed to speak.

## 3. Human review + the no-go list

A ranked score is a **suggestion**. Before anyone edits, publishes, or deletes a page, a person must
check facts the data can't see — and a list of actions must **never** be automated off this score
alone. Two code blocks below: one computes the review counts, one lays out the no-go list.

In [3]:
# Counts a human must keep in mind while reviewing
recently_updated = int((df.days_since_update < 30).sum())            # refreshed within ~a month
def truthy(s):
    return s.fillna(False).astype(bool) if s.dtype != bool else s.astype(bool)
deleted_or_unpublished = int((truthy(df.is_deleted) | ~truthy(df.is_published)).sum())
high_traffic_flagged = int(((df.score > 0) & (df.imp_b >= 60000)).sum())  # the rule's known blind spot

print("Human-review context counts (all eligible pages):")
print("  already updated within ~30 days :", recently_updated)
print("  flagged AND very high traffic   :", high_traffic_flagged, "(rule blind spot - verify each one)")
print("  deleted or unpublished          :", deleted_or_unpublished)

print("\nPer-pick human checklist (do this before acting on any row):")
checklist = [
    "Is the page still live and indexable? (data can be stale)",
    "Was it already refreshed in the last ~30 days? (don't double-edit)",
    "Is the current average position actually weak, or did traffic skew it?",
    "Does the page still earn the demand that imp_b implies? (demand can move)",
    "Is this a page we are allowed/planning to touch?",
]
for i, c in enumerate(checklist, 1):
    print(f"  {i}. {c}")

Human-review context counts (all eligible pages):
  already updated within ~30 days : 90484
  flagged AND very high traffic   : 18 (rule blind spot - verify each one)
  deleted or unpublished          : 44

Per-pick human checklist (do this before acting on any row):
  1. Is the page still live and indexable? (data can be stale)
  2. Was it already refreshed in the last ~30 days? (don't double-edit)
  3. Is the current average position actually weak, or did traffic skew it?
  4. Does the page still earn the demand that imp_b implies? (demand can move)
  5. Is this a page we are allowed/planning to touch?


**The no-go list — never automate these off this score alone:**

1. **Auto-publish or auto-rewrite content.** The playbook never proves an edit raises rankings; a
   human owns the edit decision.
2. **Auto-delete any page.** `declined` here means "impressions dropped next window", not "remove
   this page." Deleting is a permanent business call.
3. **Blindly act on `low_signals` pages.** Absence of the three signals means "off the rule," not
   "no opportunity"; a low-signals page can be a young rising page.
4. **Apply the score outside its snapshot.** Never run this on a fresh month's data as if the
   thresholds still held — re-run the window, don't reuse the numbers.
5. **Trust a top pick without reading it.** The ~15% of top-20 misses are high-traffic pages that
   held steady; only a person can tell those apart.

**Why these matter:** everything this internship validated is *association on one snapshot*. The
moment you wire the score into an automated action loop, the missing causal design becomes a real
risk instead of a footnote.

## 4. Monitoring / retrain triggers

The playbook has a shelf life. These triggers say *it's stale — rebuild it*, each with the value that
would currently satisfy it.

In [4]:
monitors = [
    ("Data contract shifted",
     "A new warehouse build moves t / B / F windows (ML-04 fixed t = 2026-05-31). Revalidation must be re-run.",
     "current t = " + str(t)),
    ("Base rate drift",
     "If the decline base rate moves far from ~0.67 on a fresh window, the label profile changed.",
     "base rate = " + str(round(lbl.mean(), 4))),
    ("Held-out precision decays",
     "Re-check precision@20 on a client-grouped split; a drop well below ~0.85 in-sample (or ~0.74 honest) means the rule weakened.",
     "p@20(in-sample) = " + str(round(p20, 3))),
    ("New-client generalization slips",
     "Grouped (new-client) AUC ~0.61 is the honest floor; a big slide means the population changed more than the model can carry.",
     "grouped AUC ~0.61 (ML-08)"),
    ("Feature distribution shifts",
     "If age or position distributions change a lot (e.g. a client backfill), thresholds 600/180/12 may no longer split the population.",
     "thresholds are settings, not laws"),
    ("Population / eligibility changes",
     "If far fewer pages meet imp_b>=100 & gsc_days_b>=15, the reviewed universe shrank.",
     "eligible = " + str(len(df))),
]
print("Trigger -> why it matters -> current read:")
for name, why, cur in monitors:
    print(f"  {name}")
    print(f"      why : {why}")
    print(f"      read: {cur}")

Trigger -> why it matters -> current read:
  Data contract shifted
      why : A new warehouse build moves t / B / F windows (ML-04 fixed t = 2026-05-31). Revalidation must be re-run.
      read: current t = 2026-05-31
  Base rate drift
      why : If the decline base rate moves far from ~0.67 on a fresh window, the label profile changed.
      read: base rate = 0.6744
  Held-out precision decays
      why : Re-check precision@20 on a client-grouped split; a drop well below ~0.85 in-sample (or ~0.74 honest) means the rule weakened.
      read: p@20(in-sample) = 0.85
  New-client generalization slips
      why : Grouped (new-client) AUC ~0.61 is the honest floor; a big slide means the population changed more than the model can carry.
      read: grouped AUC ~0.61 (ML-08)
  Feature distribution shifts
      why : If age or position distributions change a lot (e.g. a client backfill), thresholds 600/180/12 may no longer split the population.
      read: thresholds are settings, not laws
 

**Reading (section 4):** the cheapest and most important trigger is *#1 — a new warehouse build
shifts the date windows and the label, so *every* downstream number in this repo is tied to one
build, not forever. The other triggers are metric-level: watch the base rate, watch held-out
precision, watch new-client AUC. None require fancy infra — a monthly re-run of the same pipeline
against a fresh build is the whole playbook.

## 5. Exports for the paper

Write the artifacts `capstone_report.md` and `w07` pages will reuse: the ranked action queue and two
figures (reason-code breakdown, precision@K vs base-rate lift). All land in `work/outputs/`.

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

out = os.path.abspath("../outputs")
os.makedirs(out, exist_ok=True)

# 1) ranked queue (gitignored CSV - human/paper reuse, never committed)
queue.to_csv(os.path.join(out, "action_playbook_queue.csv"), index=False)
print("wrote work/outputs/action_playbook_queue.csv:", queue.shape)

# 2) figure: reason-code distribution (flagged pages)
flag = df[df.score > 0]
rc = flag.reason.value_counts().head(5)
fig, ax = plt.subplots(figsize=(7, 4))
rc.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_title("Flags by reason code (ranked queue, score > 0)")
ax.set_ylabel("pages flagged")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(os.path.join(out, "playbook_reason_codes.png"), dpi=110)
plt.close(fig)
print("wrote work/outputs/playbook_reason_codes.png")

# 3) figure: precision@K vs base rate
ks = [10, 20, 50, 100, 200, 500]
pks = [precision_at_k(queue.score.values, lbl, k) for k in ks]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, pks, marker="o", label="precision@K (rule)")
ax.axhline(lbl.mean(), color="gray", linestyle="--", label="base rate %.2f" % lbl.mean())
ax.set_xlabel("K (top ranked pages)")
ax.set_ylabel("precision")
ax.set_title("Ranked-queue precision@K vs base rate")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(out, "playbook_precision_at_k.png"), dpi=110)
plt.close(fig)
print("wrote work/outputs/playbook_precision_at_k.png")

import os as _os
for f in sorted(_os.listdir(out)):
    print("  -", f)

wrote work/outputs/action_playbook_queue.csv: (108254, 9)


wrote work/outputs/playbook_reason_codes.png
wrote work/outputs/playbook_precision_at_k.png
  - action_playbook_queue.csv
  - baseline_action_score.csv
  - playbook_precision_at_k.png
  - playbook_reason_codes.png


**Reading (section 5):** the repo rule is *no datasets in git*, so the queue CSV is written only to
the gitignored `work/outputs/` (and reused by the capstone + paper page) — never committed. The two
PNGs are small and reusable in the submitted report. These are the numbers the paper will cite:
a ranked queue of **12,930** pages, precision@20 **~0.85** (in-sample) vs base **0.674**.

## Self-check

- [x] Section 1: ranked queue with reason codes AND plain-human-readable actions
- [x] Section 2: who/for-what, plus fenced, measured limits (snapshot, ranks-not-causes, new-client)
- [x] Section 3: human review checklist + an explicit no-go list (no auto-publish/delete/blind action)
- [x] Section 4: concrete monitoring / retrain triggers with current readings
- [x] Section 5: queue CSV + two reusable figures written to `work/outputs/` (CSV gitignored)
- [x] No client names, URLs, provider values, or raw identifiers in any output
- [x] Claims use careful words: observed, measured, ranked, decision-support; no causal claims
- [ ] The notebook runs top to bottom with no errors (Kernel → Restart & Run all)
- [ ] Committed to `work/notebooks/` — then submit repo URL on the ML-10 card